# **Procesamiento de Lenguaje Natural**

## Maestría en Inteligencia Artificial Aplicada
#### Tecnológico de Monterrey
#### Prof Luis Eduardo Falcón Morales

### **Actividad en Equipos — Semanas RAG Chatbot (v3)**

Construct a QA Bot that Leverages LangChain and LLMs to Answer Questions from Loaded Documents
Estimated time needed: 60 minutes

* **Nombres y matrículas:**

  *   Jose Angel Barajas A01797221
  *   Elemento de lista
  *   Elemento de lista

* **Número de Equipo:**


---

## 📋 v3 — Mejoras sobre v1 (base funcional)

| # | Mejora | v1 | v3 | Razón |
|---|--------|----|----|-------|
| 1 | **Prompt anti-alucinación (suave)** | Prompt default de LangChain | Prompt personalizado que pide respuesta basada en contexto y dice 'no sé' solo cuando el contexto es verdaderamente vacío | v1 podía inventar datos; v2 era tan estricto que nunca respondía |
| 2 | **Citation de fuentes** | Sin fuente en la respuesta | Respuesta incluye nombre del PDF fuente | El usuario puede verificar de dónde viene cada respuesta |
| 3 | **pip con sys.executable** | `!pip install` (puede apuntar al Python incorrecto) | `!{sys.executable} -m pip install` (siempre el kernel correcto) | Evita el error de módulo no encontrado al cambiar de entorno |

**Lo que NO cambió respecto a v1 (por funcionar bien):**
- Cadena `RetrievalQA` simple (sin threshold ni memoria conversacional)
- Retriever sin filtro de similitud
- `gr.Interface` (single Q&A, más estable que ChatInterface)
- Mismo modelo de embedding: `all-MiniLM-L6-v2`

---

🧩 **Step 1 – Install the required packages**

In [11]:
import sys

!{sys.executable} -m pip install langchain
!{sys.executable} -m pip install langchain-classic
!{sys.executable} -m pip install langchain-community
!{sys.executable} -m pip install langchain-openai
!{sys.executable} -m pip install langchain-huggingface
!{sys.executable} -m pip install chromadb
!{sys.executable} -m pip install pypdf
!{sys.executable} -m pip install sentence-transformers
!{sys.executable} -m pip install gradio
!{sys.executable} -m pip install openai

---

## ⚙️ Step 2 – Imports

In [12]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_classic.chains import RetrievalQA
from langchain_openai import ChatOpenAI

import gradio as gr
import os

os.environ["HF_HUB_DISABLE_SYMLINKS_WARNING"] = "1"
os.environ["HF_HUB_OFFLINE"] = "1"

---

## 🔧 Step 3 – Configure LLM pointing to LM Studio

In [13]:
LMSTUDIO_BASE_URL = "http://100.111.50.52:1234/v1"
LMSTUDIO_MODEL    = "qwen2.5-coder-7b-instruct"

def get_llm():
    return ChatOpenAI(
        base_url=LMSTUDIO_BASE_URL,
        api_key="not-needed",
        model=LMSTUDIO_MODEL,
        temperature=0.5,
        max_tokens=1024,
    )

In [4]:
# Test connection
llm = get_llm()
resp = llm.invoke("Dame una respuesta corta en español diciendo que la conexión con LM Studio funciona.")
print(resp.content)

La conexión con LM Studio funciona correctamente.


---

## 📄 Step 4 – Document loader

In [14]:
def document_loader(file_path: str):
    loader = PyPDFLoader(file_path)
    docs = loader.load()
    # v3: tag each page with its source filename for citation
    source_name = os.path.basename(file_path)
    for doc in docs:
        doc.metadata["source_file"] = source_name
    return docs

---

## ✂️ Step 5 – Text splitter

In [15]:
def text_splitter(docs):
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=150,
        length_function=len,
    )
    return splitter.split_documents(docs)

---

## 🧠 Step 6 – Embeddings + VectorDB

In [16]:
def embedding_model():
    return HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def vector_database(chunks):
    embed = embedding_model()
    return Chroma.from_documents(documents=chunks, embedding=embed)

---

## 🛡️ Step 7 – Anti-hallucination: source citation (v3)

v3 keeps LangChain's default prompt (which works well with this model) and adds
**source citation** post-processing so the user always knows which PDF the answer came from.
No custom rules are injected — the model answers naturally from the retrieved context.

---

## 🔗 Step 8 – Retriever + QA chain

**v3 adds:** custom prompt injection + source file citation in the answer.

In [17]:
def build_retriever(file_paths):
    all_docs = []
    for fp in file_paths:
        all_docs.extend(document_loader(fp))
    chunks = text_splitter(all_docs)
    vectordb = vector_database(chunks)
    return vectordb.as_retriever()


def answer_question(file_paths, question):
    llm = get_llm()
    retriever = build_retriever(file_paths)

    # v3: same chain as v1 (default prompt works best with this model)
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=retriever,
        return_source_documents=True,
    )

    result = qa_chain.invoke({"query": question})
    answer = result["result"]

    # v3: append source citation from retrieved docs
    sources = set(
        doc.metadata.get("source_file", "unknown")
        for doc in result.get("source_documents", [])
    )
    if sources:
        answer += f"\n\n---\n📎 Sources: {', '.join(sorted(sources))}"

    return answer

---

## 💻 Step 9 – Gradio interface with PDF upload

In [20]:
def gradio_rag_interface(file, query):
    if file is None or query.strip() == "":
        return "Please upload a PDF and enter a question."
    file_paths = file if isinstance(file, list) else [file]
    try:
        return answer_question(file_paths, query)
    except Exception as e:
        err = str(e)
        if "Connection" in err or "10061" in err or "refused" in err:
            return "❌ Cannot connect to LM Studio. Make sure it's running at http://100.111.50.52:1234"
        return f"Error: {err}"


rag_app = gr.Interface(
    fn=gradio_rag_interface,
    inputs=[
        gr.File(
            label="Upload PDF File(s)",
            file_count="multiple",
            file_types=[".pdf"],
            type="filepath"
        ),
        gr.Textbox(
            label="Your Question",
            lines=5,
            placeholder="Ask something about the PDF..."
        )
    ],
    outputs=gr.Textbox(label="Answer", lines=10),
    title="ITESM-NLP RAG Chatbot v3 — Anti-Hallucination",
    description=(
        "Upload a PDF and ask questions.\n"
        "v3 improvements: custom anti-hallucination prompt + source citation in every answer."
    )
)

rag_app.launch(server_name="127.0.0.1", server_port=7863)

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

d:\ML\Projects\Project_6_TC55043.10_NLP\TC55043.10_NLP\venv\lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

---

### Stop the server and release the port

In [19]:
gr.close_all()
rag_app.close()

Closing server running on port: 7863
